In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')

columns = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
df = pd.read_csv('housing.csv', sep=r'\s+', header=None, names=columns)
X_orig = df.drop(columns=['MEDV'])
y_orig = df['MEDV']

print("="*60)
print(" PROBLEM 1: HANDLING MISSING DATA (10%)")
print("="*60)

np.random.seed(42)
X_missing = X_orig.copy()
cols_to_miss = ['CRIM', 'TAX', 'RM']
n_rows = len(X_missing)
n_missing = int(0.10 * n_rows) 

for col in cols_to_miss:
    missing_indices = np.random.choice(n_rows, n_missing, replace=False)
    X_missing.loc[missing_indices, col] = np.nan

imputer_median = SimpleImputer(strategy='median')
X_median = pd.DataFrame(imputer_median.fit_transform(X_missing), columns=X_orig.columns)

X_interp = X_missing.interpolate(method='linear', limit_direction='both')

def evaluate_p1(X, y, name):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    return {
        'Dataset': name,
        'Train MSE': mean_squared_error(y_train, model.predict(X_train)),
        'Test MSE': mean_squared_error(y_test, model.predict(X_test)),
        'Train R²': r2_score(y_train, model.predict(X_train)),
        'Test R²': r2_score(y_test, model.predict(X_test))
    }

results_p1 = [
    evaluate_p1(X_orig, y_orig, 'Original Data'),
    evaluate_p1(X_median, y_orig, 'Imputed (Median)'),
    evaluate_p1(X_interp, y_orig, 'Imputed (Interpolation)')
]
df_res_p1 = pd.DataFrame(results_p1).round(3)
print(df_res_p1.to_markdown(index=False))


print("\n\n" + "="*60)
print(" PROBLEM 2: REGRESSION ON NOISY DATA")
print("="*60)

noise = np.random.normal(loc=0, scale=5.0, size=len(y_orig))
y_noisy = y_orig + noise

def evaluate_poly_regularization(X, y, title):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    degrees = [1, 2, 3]
    alphas_to_try = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    results = []

    for d in degrees:
        model_lr = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression())
        model_lr.fit(X_train, y_train)
        results.append({
            'Degree': d, 'Model': 'Linear', 'Best Alpha': '-',
            'Train MSE': mean_squared_error(y_train, model_lr.predict(X_train)),
            'Test MSE': mean_squared_error(y_test, model_lr.predict(X_test)),
            'Train R²': r2_score(y_train, model_lr.predict(X_train)),
            'Test R²': r2_score(y_test, model_lr.predict(X_test))
        })

        ridge_grid = GridSearchCV(make_pipeline(PolynomialFeatures(d), StandardScaler(), Ridge()),
                                    param_grid={'ridge__alpha': alphas_to_try}, cv=5)
        ridge_grid.fit(X_train, y_train)
        best_ridge = ridge_grid.best_estimator_
        results.append({
            'Degree': d, 'Model': 'Ridge', 'Best Alpha': ridge_grid.best_params_['ridge__alpha'],
            'Train MSE': mean_squared_error(y_train, best_ridge.predict(X_train)),
            'Test MSE': mean_squared_error(y_test, best_ridge.predict(X_test)),
            'Train R²': r2_score(y_train, best_ridge.predict(X_train)),
            'Test R²': r2_score(y_test, best_ridge.predict(X_test))
        })

        lasso_grid = GridSearchCV(make_pipeline(PolynomialFeatures(d), StandardScaler(), Lasso(max_iter=5000)),
                                    param_grid={'lasso__alpha': alphas_to_try}, cv=5)
        lasso_grid.fit(X_train, y_train)
        best_lasso = lasso_grid.best_estimator_
        results.append({
            'Degree': d, 'Model': 'Lasso', 'Best Alpha': lasso_grid.best_params_['lasso__alpha'],
            'Train MSE': mean_squared_error(y_train, best_lasso.predict(X_train)),
            'Test MSE': mean_squared_error(y_test, best_lasso.predict(X_test)),
            'Train R²': r2_score(y_train, best_lasso.predict(X_train)),
            'Test R²': r2_score(y_test, best_lasso.predict(X_test))
        })

    print(f"\n--- {title} ---")
    df_res = pd.DataFrame(results).round(3)
    print(df_res.to_markdown(index=False))

evaluate_poly_regularization(X_orig, y_orig, "TABLE 1: WITHOUT NOISE")
evaluate_poly_regularization(X_orig, y_noisy, "TABLE 2: WITH NOISE")

 PROBLEM 1: HANDLING MISSING DATA (10%)
| Dataset                 |   Train MSE |   Test MSE |   Train R² |   Test R² |
|:------------------------|------------:|-----------:|-----------:|----------:|
| Original Data           |      21.641 |     24.291 |      0.751 |     0.669 |
| Imputed (Median)        |      23.539 |     24.269 |      0.729 |     0.669 |
| Imputed (Interpolation) |      22.688 |     23.417 |      0.739 |     0.681 |


 PROBLEM 2: REGRESSION ON NOISY DATA

--- TABLE 1: WITHOUT NOISE ---
|   Degree | Model   | Best Alpha   |   Train MSE |   Test MSE |   Train R² |   Test R² |
|---------:|:--------|:-------------|------------:|-----------:|-----------:|----------:|
|        1 | Linear  | -            |      21.641 |     24.291 |      0.751 |     0.669 |
|        1 | Ridge   | 1.0          |      21.643 |     24.313 |      0.751 |     0.668 |
|        1 | Lasso   | 0.001        |      21.641 |     24.295 |      0.751 |     0.669 |
|        2 | Linear  | -            |  